<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/ml/notebooks/c5_l8.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C5-L8 · Deployment: predict_demo.py
Del notebook al sistema: validación con falla cerrada, mismo featurizador en train y serve, predicción como JSON.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/ml/data/c5_l8.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c5_l8.csv'), Path('data/c5_l8.csv'), Path('c5_l8.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)
print(df.head(5).to_string(index=False))

In [ ]:
df['ret'] = df['close'].pct_change()
df['rango'] = (df['high']-df['low'])/df['close']
for k in (1, 2, 3, 5):
    df[f'lag_{k}'] = df['ret'].shift(k)
df['mom5'] = df['close']/df['close'].shift(5) - 1
data = df.dropna().reset_index(drop=True)
FEAT = ['rango','lag_1','lag_2','lag_3','lag_5','mom5']
X = data[FEAT].values
y = (data['ret'].shift(-1).fillna(0) > 0).astype(int).values[:-1]
X_train = X[:-1]
from sklearn.linear_model import RidgeClassifier
modelo = RidgeClassifier().fit(X_train, y)
print('entrenado en', len(X_train), 'filas; mom5 última barra =', round(float(X[-1, 5]), 6))

In [ ]:
import urllib.request, subprocess, sys, json
from pathlib import Path
SRC = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/ml/predict_demo.py'
try:
    blob = urllib.request.urlopen(SRC, timeout=10).read()
    script = Path('predict_demo_dl.py'); script.write_bytes(blob); print('script: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    script = Path('../predict_demo.py')
csv = [c for c in [Path('../data/c5_l8.csv'), Path('data/c5_l8.csv'), Path('c5_l8.csv')] if c.exists()][0]
r = subprocess.run([sys.executable, str(script), '--input', str(csv)], capture_output=True, text=True)
print('returncode:', r.returncode)
print(r.stdout.strip())
print(r.stderr[-500:] if r.returncode != 0 else '(sin errores)')
out = json.loads(r.stdout.strip())

In [ ]:
import tempfile, os
bad = data[['dia','close','high','low']].copy()  # sin 'volumen': debe fallar cerrado
fd, bad_path = tempfile.mkstemp(suffix='.csv')
os.close(fd)
bad.to_csv(bad_path, index=False)
r2 = subprocess.run([sys.executable, str(script), '--input', bad_path], capture_output=True, text=True)
print('returncode:', r2.returncode)
print(r2.stdout.strip())
os.unlink(bad_path)

In [ ]:
assert r.returncode == 0 and out.get('ok') is True
assert out['pred'] in (0, 1) and isinstance(out['filas'], int) and out['filas'] >= 30
assert r2.returncode != 0 and json.loads(r2.stdout.strip()).get('ok') is False
print(f"OK L8: deployment verificado, pred={out['pred']}, falla cerrada OK")